# OE2 · Cascade Mask R-CNN (R50-FPN) · FASE A

Se entrena y evalúa Cascade Mask R-CNN con Detectron2 como parte de la comparación de modelos del OE2.

## 1) Comprobar la GPU y conectar Google Drive
Se verifica que haya GPU disponible y se monta Google Drive.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
assert torch.cuda.is_available(), "Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU (A100)"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

from google.colab import drive
drive.mount("/content/drive")

GPU: NVIDIA A100-SXM4-40GB | torch 2.11.0+cu130 | CUDA 13.0
Mounted at /content/drive


## 2) Instalar Detectron2
Se instala desde un commit fijo para que los resultados sean reproducibles.

In [2]:
%pip install -q pycocotools
%pip install -q 'git+https://github.com/facebookresearch/detectron2.git@02b5c4e295e990042a714712c21dc79b731e8833'
import detectron2
print("detectron2", detectron2.__version__)

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 517.0 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 14.7 MB/s eta 0:00:00
detectron2 0.6


## 3) Configuración
Se define la semilla y el dataset de entrenamiento.

In [3]:
import json

SEED = 42

PROYECTO = "/content/drive/MyDrive/tesis-rg6final"
DATASET = "data_estratificado1"

ARCH_TAG = "cascade_maskrcnn"
ARCH_PRINT = "Cascade Mask R-CNN (R50-FPN)"
CONFIG_YAML = "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"

DRIVE_OUT = os.path.join(PROYECTO, "resultados", "oe2", "cascade", f"seed_{SEED}")
REPORT_DIR = os.path.join(DRIVE_OUT, "reportes")
DRIVE_CKPT = os.path.join(DRIVE_OUT, "ckpt")
DATASET_DIR = "/content/datos"
OUTPUT_DIR = f"/content/{ARCH_TAG}_entrenamiento"
os.makedirs(REPORT_DIR, exist_ok=True)

with open(os.path.join(PROYECTO, DATASET, "origen.json"), encoding="utf-8") as f:
    ORIGEN = json.load(f)
print(f"{ARCH_PRINT} | semilla {SEED} | datos {DATASET} ({ORIGEN['entrada']}) {ORIGEN['imagenes']}")
print("Salida:", DRIVE_OUT)

Cascade Mask R-CNN (R50-FPN) | semilla 42 | datos data_estratificado1 (data_curado) {'train': 982, 'valid': 281, 'test': 141}
Salida: /content/drive/MyDrive/tesis-rg6final/resultados/oe2/cascade/seed_42


## 4) Copiar el dataset y verificar que no haya imágenes repetidas
Se copia el dataset al disco de Colab y se comprueba con MD5 que ninguna imagen aparezca en más de una partición.

In [4]:
import shutil, hashlib, collections

def md5(ruta):
    with open(ruta, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

if not os.path.exists(DATASET_DIR):
    shutil.copytree(os.path.join(PROYECTO, DATASET), DATASET_DIR)

contenido = {}
for parte in ["train", "valid", "test"]:
    with open(os.path.join(DATASET_DIR, parte, "_annotations.coco.json"), encoding="utf-8") as f:
        coco = json.load(f)
    contenido[parte] = {md5(os.path.join(DATASET_DIR, parte, im["file_name"])) for im in coco["images"]}
    por_clase = collections.Counter(a["category_id"] for a in coco["annotations"])
    print(f"{parte:5s}: {len(coco['images'])} fotos | {len(coco['annotations'])} anotaciones | por clase {dict(por_clase)}")
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    repetidas = contenido[a] & contenido[b]
    assert not repetidas, f"FUGA: {len(repetidas)} fotos idénticas entre {a} y {b}. Corrígelo antes de entrenar."
print("Sin fugas entre train/valid/test")

train: 982 fotos | 4445 anotaciones | por clase {1: 2233, 0: 2212}
valid: 281 fotos | 1309 anotaciones | por clase {0: 660, 1: 649}
test : 141 fotos | 667 anotaciones | por clase {0: 349, 1: 318}
Sin fugas entre train/valid/test


## 5) Receta de entrenamiento
| Parámetro | Valor | Por qué |
|---|---|---|
| Pesos iniciales | COCO (model zoo) | *fine-tuning* desde un modelo ya entrenado |
| Lote (`IMS_PER_BATCH`) | 8 | cabe holgado en la A100 (se midió ~6.5 de 40 GB) |
| `BASE_LR` | 0.0005 | 0.00025 con lote 4 → ×2 al duplicar el lote (escalado lineal, Goyal et al.) |
| Épocas | 50 → `MAX_ITER = ceil(n_train / 8) × 50` | mismo presupuesto que YOLO |
| Calentamiento | min(1000, 5 % de las iteraciones) | arranque suave de la LR |
| Bajadas de LR | ×0.1 al 70 % y al 90 % | calendario estándar de Detectron2 |
| Resolución | lado corto 768, máximo 1024 | igual que YOLO `imgsz=1024` (una foto 4:3 queda en 1024×768) |
| Evaluación en valid | cada `max(200, 5 épocas)` iteraciones | se guarda el mejor `segm/AP` como `model_best.pth` |
| Umbral / detecciones | 0.001 / 300 por imagen | protocolo COCO común a los tres modelos |
| AMP + TF32 | sí | más rápido en la A100, sin efecto práctico en el mAP |

In [5]:
import math

EPOCHS = 50
IMS_PER_BATCH = 8
BASE_LR = 0.0005
NUM_WORKERS = 8
WARMUP_RATIO, WARMUP_CAP = 0.05, 1000
MIN_SIZE_TRAIN, MAX_SIZE_TRAIN = 768, 1024
MIN_SIZE_TEST, MAX_SIZE_TEST = 768, 1024
SCORE_THRESH_TEST = 0.001
DETECTIONS_PER_IMAGE = 300
BEST_METRIC = "segm/AP"

## 6) Preparar Detectron2
Se registran los datasets, el aumento de datos y el entrenador.

In [6]:
import shutil
from detectron2.utils.logger import setup_logger
setup_logger()
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, DatasetMapper, build_detection_train_loader
from detectron2.data import transforms as T
from detectron2.data.datasets import register_coco_instances
from detectron2.engine import DefaultTrainer, HookBase, hooks
from detectron2.evaluation import COCOEvaluator
from detectron2.utils.env import seed_all_rng
from torch.amp import autocast, GradScaler

def ensure_coco_info(ruta_json):
    with open(ruta_json) as f:
        d = json.load(f)
    if "info" not in d or "licenses" not in d:
        d.setdefault("info", {"description": "dataset", "version": "1.0"})
        d.setdefault("licenses", [])
        with open(ruta_json, "w") as f:
            json.dump(d, f)

for parte in ["train", "valid", "test"]:
    ensure_coco_info(os.path.join(DATASET_DIR, parte, "_annotations.coco.json"))
    if f"rg6_{parte}" not in DatasetCatalog.list():
        register_coco_instances(f"rg6_{parte}", {}, os.path.join(DATASET_DIR, parte, "_annotations.coco.json"),
                                os.path.join(DATASET_DIR, parte))

with open(os.path.join(DATASET_DIR, "train", "_annotations.coco.json")) as f:
    CLASES = [c["name"] for c in sorted(json.load(f)["categories"], key=lambda c: c["id"])]
print("Clases:", CLASES)

def aumento_common_a():
    return [
        T.ResizeShortestEdge(short_edge_length=[MIN_SIZE_TRAIN], max_size=MAX_SIZE_TRAIN, sample_style="choice"),
        T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
        T.RandomBrightness(0.85, 1.15),
        T.RandomContrast(0.85, 1.15),
        T.RandomSaturation(0.90, 1.10),
        T.RandomRotation(angle=[-5, 5], expand=False, center=None, sample_style="range"),
    ]

class SyncDriveHook(HookBase):
    def __init__(self, periodo):
        self.periodo = periodo

    def copiar(self):
        os.makedirs(DRIVE_CKPT, exist_ok=True)
        ruta_last = os.path.join(OUTPUT_DIR, "last_checkpoint")
        if os.path.exists(ruta_last):
            with open(ruta_last) as f:
                ultimo = f.read().strip()
            if os.path.exists(os.path.join(OUTPUT_DIR, ultimo)):
                shutil.copy2(os.path.join(OUTPUT_DIR, ultimo), DRIVE_CKPT)
                shutil.copy2(ruta_last, DRIVE_CKPT)
                for viejo in os.listdir(DRIVE_CKPT):
                    if viejo.startswith("model_0") and viejo.endswith(".pth") and viejo != ultimo:
                        os.remove(os.path.join(DRIVE_CKPT, viejo))
        for archivo in ("model_best.pth", "metrics.json"):
            if os.path.exists(os.path.join(OUTPUT_DIR, archivo)):
                shutil.copy2(os.path.join(OUTPUT_DIR, archivo), DRIVE_CKPT)
        mejor = next((h for h in self.trainer._hooks if isinstance(h, hooks.BestCheckpointer)), None)
        if mejor is not None:
            with open(os.path.join(DRIVE_CKPT, "best_state.json"), "w") as f:
                json.dump({"best_metric": mejor.best_metric, "best_iter": mejor.best_iter}, f)

    def after_step(self):
        if (self.trainer.iter + 1) % self.periodo == 0:
            self.copiar()

    def after_train(self):
        self.copiar()

class FairTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        carpeta = output_folder or os.path.join(cfg.OUTPUT_DIR, "inference")
        os.makedirs(carpeta, exist_ok=True)
        return COCOEvaluator(dataset_name, cfg, distributed=False, output_dir=carpeta)

    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=DatasetMapper(cfg, is_train=True, augmentations=aumento_common_a()))

    def build_hooks(self):
        lista = super().build_hooks()
        lista.insert(-1, hooks.BestCheckpointer(self.cfg.TEST.EVAL_PERIOD, self.checkpointer, BEST_METRIC,
                                                mode="max", file_prefix="model_best"))
        lista.append(SyncDriveHook(self.cfg.TEST.EVAL_PERIOD))
        return lista

    def run_step(self):
        assert self.model.training
        if not hasattr(self, "_grad_scaler"):
            self._grad_scaler = GradScaler("cuda")
        interno = self._trainer
        if not hasattr(interno, "_data_loader_iter") or interno._data_loader_iter is None:
            interno._data_loader_iter = iter(interno.data_loader)
        datos = next(interno._data_loader_iter)
        with autocast("cuda"):
            perdidas = self.model(datos)
            perdida_total = sum(perdidas.values())
        self.optimizer.zero_grad(set_to_none=True)
        self._grad_scaler.scale(perdida_total).backward()
        self._grad_scaler.step(self.optimizer)
        self._grad_scaler.update()
        self.storage.put_scalars(total_loss=float(perdida_total.item()),
                                 **{k: float(v.item()) for k, v in perdidas.items()})

/usr/local/lib/python3.13/dist-packages/detectron2/model_zoo/model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Clases: ['rg6Bien', 'rg6Mal']


## 7) Entrenar
Se entrena el modelo guardando puntos de control para poder reanudar.

In [ ]:
n_train = len(DatasetCatalog.get("rg6_train"))
iters_por_epoca = math.ceil(n_train / IMS_PER_BATCH)
max_iter = iters_por_epoca * EPOCHS
warmup_iters = min(WARMUP_CAP, int(max_iter * WARMUP_RATIO))
eval_period = max(200, iters_por_epoca * 5)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(CONFIG_YAML))
cfg.DATASETS.TRAIN = ("rg6_train",)
cfg.DATASETS.TEST = ("rg6_valid",)
cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS
cfg.INPUT.MIN_SIZE_TRAIN = [MIN_SIZE_TRAIN]
cfg.INPUT.MAX_SIZE_TRAIN = MAX_SIZE_TRAIN
cfg.INPUT.MIN_SIZE_TEST = MIN_SIZE_TEST
cfg.INPUT.MAX_SIZE_TEST = MAX_SIZE_TEST
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASES)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST
cfg.TEST.DETECTIONS_PER_IMAGE = DETECTIONS_PER_IMAGE
cfg.SOLVER.IMS_PER_BATCH = IMS_PER_BATCH
cfg.SOLVER.BASE_LR = BASE_LR
cfg.SOLVER.MAX_ITER = max_iter
cfg.SOLVER.WARMUP_ITERS = warmup_iters
cfg.SOLVER.STEPS = (int(max_iter * 0.7), int(max_iter * 0.9))
cfg.SOLVER.GAMMA = 0.1
cfg.TEST.EVAL_PERIOD = eval_period
cfg.SOLVER.CHECKPOINT_PERIOD = eval_period
cfg.SOLVER.AMP.ENABLED = True
cfg.SEED = SEED
cfg.OUTPUT_DIR = OUTPUT_DIR
cfg.MODEL.DEVICE = "cuda"
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(CONFIG_YAML)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Train: {n_train} fotos | {iters_por_epoca} iter/época | MAX_ITER={max_iter} | warmup={warmup_iters} | "
      f"eval cada {eval_period}")

PESOS = os.path.join(DRIVE_OUT, f"{ARCH_TAG}_model_best.pth")
if os.path.exists(os.path.join(DRIVE_OUT, "TERMINADO.json")):
    print("Esta semilla ya terminó de entrenar: se salta.")
else:
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    seed_all_rng(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    reanudar = os.path.exists(os.path.join(DRIVE_CKPT, "last_checkpoint"))
    if reanudar:
        for archivo in os.listdir(DRIVE_CKPT):
            shutil.copy2(os.path.join(DRIVE_CKPT, archivo), OUTPUT_DIR)
        print("REANUDO desde", DRIVE_CKPT)

    trainer = FairTrainer(cfg)
    trainer.resume_or_load(resume=reanudar)
    if reanudar and os.path.exists(os.path.join(DRIVE_CKPT, "best_state.json")):
        with open(os.path.join(DRIVE_CKPT, "best_state.json")) as f:
            estado = json.load(f)
        for h in trainer._hooks:
            if isinstance(h, hooks.BestCheckpointer):
                for k, v in estado.items():
                    if v is not None:
                        setattr(h, k, v)
        print("Mejor modelo hasta ahora:", estado)
    trainer.train()

    mejor = os.path.join(OUTPUT_DIR, "model_best.pth")
    if not os.path.exists(mejor):
        mejor = os.path.join(OUTPUT_DIR, "model_final.pth")
        print("No hay model_best.pth: se usa model_final.pth")
    shutil.copy2(mejor, PESOS)
    if ARCH_TAG == "maskrcnn":
        shutil.copy2(mejor, os.path.join(DRIVE_OUT, "model_best.pth"))
    os.makedirs(os.path.join(DRIVE_OUT, "artefactos", "entrenamiento"), exist_ok=True)
    for archivo in ("metrics.json", "config.yaml"):
        if os.path.exists(os.path.join(OUTPUT_DIR, archivo)):
            shutil.copy2(os.path.join(OUTPUT_DIR, archivo), os.path.join(DRIVE_OUT, "artefactos", "entrenamiento"))

    manifiesto = {"arch": ARCH_PRINT, "profile": "A100_MAX", "seed": SEED, "dataset": DATASET,
                  "huella_datos": ORIGEN["huella"], "ims_per_batch": IMS_PER_BATCH, "grad_accum": 1,
                  "batch_efectivo": IMS_PER_BATCH, "workers": NUM_WORKERS, "deterministic": False,
                  "pasos_optimizador": max_iter, "base_lr": BASE_LR, "epochs": EPOCHS, "max_iter": max_iter,
                  "warmup_iters": warmup_iters, "eval_period": eval_period, "tf32": True, "amp": True,
                  "detections_per_image": DETECTIONS_PER_IMAGE,
                  "min_size_train": MIN_SIZE_TRAIN, "max_size_train": MAX_SIZE_TRAIN,
                  "min_size_test": MIN_SIZE_TEST, "max_size_test": MAX_SIZE_TEST, "reanudado": reanudar,
                  "torch": torch.__version__, "detectron2": detectron2.__version__,
                  "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0), "best": PESOS}
    for destino in (DRIVE_OUT, os.path.join(DRIVE_OUT, "artefactos", "entrenamiento")):
        with open(os.path.join(destino, "run_manifest.json"), "w") as f:
            json.dump(manifiesto, f, indent=2)
    with open(os.path.join(DRIVE_OUT, "TERMINADO.json"), "w") as f:
        json.dump({"best": PESOS}, f)
    print("Entrenamiento terminado. Pesos:", PESOS)

WARNING [10/20 05:50:56 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[10/20 05:50:56 d2.data.datasets.coco]: Loaded 982 images in COCO format from /content/datos/train/_annotations.coco.json
Train: 982 fotos | 123 iter/época | MAX_ITER=6150 | warmup=307 | eval cada 615
[10/20 05:50:57 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256,

/usr/local/lib/python3.13/dist-packages/detectron2/engine/train_loop.py:471: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  grad_scaler = GradScaler()
model_final_480dd8.pkl: 288MB [00:01, 198MB/s]                           
roi_heads.box_predictor.0.cls_score.{bias, weight}
roi_heads.box_predictor.1.cls_score.{bias, weight}
roi_heads.box_predictor.2.cls_score.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}


[10/20 05:50:59 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W1020 05:51:08.874000 743 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
/usr/local/lib/python3.13/dist-packages/detectron2/engine/hooks.py:359: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https:

[10/20 05:51:55 d2.utils.events]:  eta: 2:23:43  iter: 19  total_loss: 4.781  loss_cls_stage0: 1.067  loss_box_reg_stage0: 0.08884  loss_cls_stage1: 1.07  loss_box_reg_stage1: 0.0974  loss_cls_stage2: 1.277  loss_box_reg_stage2: 0.065  loss_mask: 0.6915  loss_rpn_cls: 0.3506  loss_rpn_loc: 0.04075    time: 1.9360  last_time: 1.2771   lr: 3.1414e-05  max_mem: 36691M
[10/20 05:52:25 d2.utils.events]:  eta: 2:20:22  iter: 39  total_loss: 3.681  loss_cls_stage0: 0.7278  loss_box_reg_stage0: 0.1657  loss_cls_stage1: 0.735  loss_box_reg_stage1: 0.1618  loss_cls_stage2: 0.8379  loss_box_reg_stage2: 0.1039  loss_mask: 0.6863  loss_rpn_cls: 0.165  loss_rpn_loc: 0.03783    time: 1.5904  last_time: 1.3910   lr: 6.3954e-05  max_mem: 36691M
[10/20 05:52:51 d2.utils.events]:  eta: 2:20:52  iter: 59  total_loss: 2.772  loss_cls_stage0: 0.4292  loss_box_reg_stage0: 0.2301  loss_cls_stage1: 0.4145  loss_box_reg_stage1: 0.2886  loss_cls_stage2: 0.4171  loss_box_reg_stage2: 0.168  loss_mask: 0.6735  loss

## 8) Evaluación en test
Se evalúa el modelo en el test con COCOeval (mAP de caja y de máscara).

In [ ]:
import io, contextlib, csv
import numpy as np
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor
from detectron2.evaluation import inference_on_dataset
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

cfg_eval = get_cfg()
cfg_eval.merge_from_file(model_zoo.get_config_file(CONFIG_YAML))
cfg_eval.DATASETS.TEST = ("rg6_test",)
cfg_eval.DATALOADER.NUM_WORKERS = 8
cfg_eval.INPUT.MIN_SIZE_TEST = MIN_SIZE_TEST
cfg_eval.INPUT.MAX_SIZE_TEST = MAX_SIZE_TEST
cfg_eval.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASES)
cfg_eval.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST
cfg_eval.TEST.DETECTIONS_PER_IMAGE = DETECTIONS_PER_IMAGE
cfg_eval.MODEL.WEIGHTS = PESOS
cfg_eval.MODEL.DEVICE = "cuda"

predictor = DefaultPredictor(cfg_eval)
carpeta_inferencia = f"/content/{ARCH_TAG}_inferencia_test"
evaluador = COCOEvaluator("rg6_test", cfg_eval, distributed=False, output_dir=carpeta_inferencia)
resultados = inference_on_dataset(predictor.model, build_detection_test_loader(cfg_eval, "rg6_test"), evaluador)
bbox, segm = resultados["bbox"], resultados["segm"]

carpeta_pred = os.path.join(DRIVE_OUT, "artefactos", "predicciones_test")
os.makedirs(carpeta_pred, exist_ok=True)
PRED_JSON = os.path.join(carpeta_pred, "coco_instances_results.json")
shutil.copy2(os.path.join(carpeta_inferencia, "coco_instances_results.json"), PRED_JSON)
GT_JSON = os.path.join(DATASET_DIR, "test", "_annotations.coco.json")

def ap_por_clase(tipo):
    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO(GT_JSON)
        coco_dt = coco_gt.loadRes(PRED_JSON)
    salida = {}
    for c in coco_gt.loadCats(coco_gt.getCatIds()):
        ev = COCOeval(coco_gt, coco_dt, iouType=tipo)
        ev.params.catIds = [c["id"]]
        with contextlib.redirect_stdout(io.StringIO()):
            ev.evaluate(); ev.accumulate(); ev.summarize()
        salida[c["name"]] = {"AP": float(ev.stats[0]), "AP50": float(ev.stats[1]), "AP75": float(ev.stats[2])}
    promedio = {k: float(np.mean([v[k] for v in salida.values()])) for k in ["AP", "AP50", "AP75"]}
    return salida, promedio

por_bbox, prom_bbox = ap_por_clase("bbox")
por_segm, prom_segm = ap_por_clase("segm")
ruta_por_clase = os.path.join(REPORT_DIR, f"obj2_{ARCH_TAG}_phaseA_per_class_ap.json")
with open(ruta_por_clase, "w") as f:
    json.dump({"bbox": por_bbox, "segm": por_segm, "avg_bbox": prom_bbox, "avg_segm": prom_segm}, f, indent=2)

import csv

def guardar_reporte(fila):
    ruta_json = os.path.join(REPORT_DIR, "obj2_results_all_models.json")
    filas = []
    if os.path.exists(ruta_json):
        with open(ruta_json) as f:
            filas = json.load(f)
    filas = [r for r in filas if r.get("arch") != fila["arch"]] + [fila]
    with open(ruta_json, "w") as f:
        json.dump(filas, f, indent=2)
    columnas = list(dict.fromkeys(k for r in filas for k in r))
    with open(os.path.join(REPORT_DIR, "obj2_results_all_models.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=columnas)
        w.writeheader()
        for r in filas:
            w.writerow({k: r.get(k, "") for k in columnas})
    print("Reporte guardado:", ruta_json)

fila = {"arch": f"{ARCH_PRINT} (Detectron2)", "phase": "A",
        "map_bbox_50_95": round(bbox["AP"], 6), "ap50_bbox": round(bbox["AP50"], 6), "ap75_bbox": round(bbox["AP75"], 6),
        "map_segm_50_95": round(segm["AP"], 6), "ap50_segm": round(segm["AP50"], 6), "ap75_segm": round(segm["AP75"], 6),
        "model_size_mb": round(os.path.getsize(PESOS) / 2**20, 3),
        "params_millones": round(sum(p.numel() for p in predictor.model.parameters()) / 1e6, 3),
        "weights": PESOS, "dataset": DATASET, "pred_json": PRED_JSON, "per_class_file": ruta_por_clase,
        "score_thresh_test": SCORE_THRESH_TEST, "max_det": DETECTIONS_PER_IMAGE,
        "evaluator": "pycocotools", "ap_scale": "0-100",
        "torch": torch.__version__, "framework": f"detectron2 {detectron2.__version__}",
        "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0), "seed": SEED, "profile": "A100_MAX"}
for extension in ("json", "csv"):
    with open(os.path.join(REPORT_DIR, f"obj2_{ARCH_TAG}_phaseA.{extension}"), "w", newline="") as f:
        if extension == "json":
            json.dump(fila, f, indent=2)
        else:
            w = csv.DictWriter(f, fieldnames=list(fila)); w.writeheader(); w.writerow(fila)
guardar_reporte(fila)

print(f"\nTEST · {ARCH_PRINT} · semilla {SEED}")
print(f"  mAP50-95 máscara: {segm['AP']:.2f} | AP50 {segm['AP50']:.2f} | AP75 {segm['AP75']:.2f}")
print(f"  mAP50-95 caja   : {bbox['AP']:.2f} | AP50 {bbox['AP50']:.2f} | AP75 {bbox['AP75']:.2f}")
print("  AP máscara por clase (0-1):", {k: round(v["AP"], 4) for k, v in por_segm.items()})

## 9) Matriz de confusión (umbral 0.25)
Se calcula la matriz de confusión a partir de las predicciones del test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as mask_util

def matriz_confusion(tag, pred_json, gt_json, out_dir, conf=0.25, iou_min=0.5):
    os.makedirs(out_dir, exist_ok=True)
    coco = COCO(gt_json)
    cats = sorted(coco.loadCats(coco.getCatIds()), key=lambda c: c["id"])
    idx_de = {c["id"]: i for i, c in enumerate(cats)}
    nombres = [c["name"] for c in cats]
    C = len(cats)
    with open(pred_json) as f:
        preds = [p for p in json.load(f) if p["score"] >= conf and "segmentation" in p]
    por_img = {}
    for p in preds:
        por_img.setdefault(p["image_id"], []).append(p)

    cm = np.zeros((C + 1, C + 1), dtype=np.int64)
    for img_id in coco.getImgIds():
        anots = [a for a in coco.loadAnns(coco.getAnnIds(imgIds=[img_id])) if not a.get("iscrowd", 0)]
        reales = [(idx_de[a["category_id"]], coco.annToMask(a).astype(bool)) for a in anots]
        usados = set()
        for p in sorted(por_img.get(img_id, []), key=lambda p: -p["score"]):
            clase_p = idx_de[p["category_id"]]
            mascara_p = mask_util.decode(p["segmentation"]).astype(bool)
            mejor, mejor_iou = -1, 0.0
            for k, (_, mascara_r) in enumerate(reales):
                if k in usados:
                    continue
                inter = np.logical_and(mascara_p, mascara_r).sum()
                if inter == 0:
                    continue
                iou = inter / float(np.logical_or(mascara_p, mascara_r).sum())
                if iou > mejor_iou:
                    mejor_iou, mejor = iou, k
            if mejor >= 0 and mejor_iou >= iou_min:
                usados.add(mejor)
                cm[reales[mejor][0], clase_p] += 1
            else:
                cm[C, clase_p] += 1
        for k, (clase_r, _) in enumerate(reales):
            if k not in usados:
                cm[clase_r, C] += 1

    etiquetas = nombres + ["background"]
    np.savetxt(os.path.join(out_dir, "confusion_matrix.csv"), cm, fmt="%d", delimiter=",")
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm)
    ax.set_xticks(range(C + 1)); ax.set_yticks(range(C + 1))
    ax.set_xticklabels(etiquetas, rotation=30, ha="right"); ax.set_yticklabels(etiquetas)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
    ax.set_title(f"CM {tag} | conf>={conf} | IoU_mask>={iou_min}")
    for i in range(C + 1):
        for j in range(C + 1):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200)
    plt.show()

    precision = {nombres[k]: round(cm[k, k] / max(cm[:, k].sum(), 1), 4) for k in range(C)}
    recall = {nombres[k]: round(cm[k, k] / max(cm[k, :].sum(), 1), 4) for k in range(C)}
    with open(os.path.join(out_dir, "confusion_summary.json"), "w") as f:
        json.dump({"tag": tag, "conf": conf, "iou_mask": iou_min, "labels": etiquetas, "cm": cm.tolist(),
                   "precision_por_clase": precision, "recall_por_clase": recall,
                   "pred_json": pred_json, "gt_json": gt_json}, f, indent=2, ensure_ascii=False)
    print("Precisión por clase:", precision)
    print("Recall por clase:   ", recall)

matriz_confusion(f"{ARCH_TAG}_phaseA", PRED_JSON, os.path.join(DATASET_DIR, "test", "_annotations.coco.json"),
                 os.path.join(REPORT_DIR, "confusion_" + f"{ARCH_TAG}_phaseA"))

## 10) Registro de la sesión
Se guarda la semilla, la GPU y la fecha de la ejecución.

In [ ]:
import datetime
with open(os.path.join(DRIVE_OUT, "sesion.json"), "w") as f:
    json.dump({"seed": SEED, "dataset": DATASET, "huella_datos": ORIGEN["huella"],
               "gpu": torch.cuda.get_device_name(0),
               "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
               "fecha_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}, f, indent=2)

for raiz, _, archivos in os.walk(DRIVE_OUT):
    if "ckpt" in raiz:
        continue
    for a in sorted(archivos):
        ruta = os.path.join(raiz, a)
        print(f"  {os.path.getsize(ruta) / 2**20:8.1f} MB  {os.path.relpath(ruta, DRIVE_OUT)}")